In [202]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import optuna
from sklearn.metrics import classification_report,accuracy_score,confusion_matrix,f1_score,make_scorer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier,AdaBoostClassifier
from sklearn.model_selection import train_test_split,RandomizedSearchCV,cross_val_score
from sklearn.preprocessing import StandardScaler,MinMaxScaler
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from imblearn.combine import SMOTEENN
from imblearn.over_sampling import SMOTE,ADASYN
from collections import Counter
import warnings

In [199]:
model_dt=DecisionTreeClassifier()
model_dt2=DecisionTreeClassifier()
model_dt3=DecisionTreeClassifier()
model_dt_smotenn=DecisionTreeClassifier()
model_rf_smotenn=RandomForestClassifier(n_estimators=500)
model_xgb=XGBClassifier(random_state=42)
model_xgb_smotenn=XGBClassifier(random_state=42)
model_xgb_smote=XGBClassifier(random_state=42)
model_xgb_adasyn=XGBClassifier(random_state=42)
model_cat=CatBoostClassifier(auto_class_weights='Balanced',verbose=0,random_state=42)
model_rf=RandomForestClassifier(class_weight='balanced',n_estimators=300,max_depth=6,random_state=42)
sc=StandardScaler()
mns=MinMaxScaler()
smenn=SMOTEENN()
smote=SMOTE()
adasyn=ADASYN(random_state=42)
warnings.filterwarnings(action='ignore')

In [109]:
df=pd.read_csv('../data/Customer-Churn.csv')

In [110]:
df.head(5)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [111]:
df.Churn.value_counts()/len(df)*100

Churn
No     73.463013
Yes    26.536987
Name: count, dtype: float64

## **Churn Rate**: 26.53%

- Which means, 26.53% of the customers churn out of this telecom company

In [112]:
y=df['Churn']
x=df.drop(columns=['customerID','Churn'])


print("Shape of X :",x.shape)
print("Shape of y",y.shape)

Shape of X : (7043, 19)
Shape of y (7043,)


## Train Test Split

In [113]:
x=pd.get_dummies(x,drop_first=True)
y=df['Churn'].map({'No':0,'Yes':1})

In [114]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2)

## Model Building

In [115]:
model_dt.fit(x_train,y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [116]:
y_pred_dt=model_dt.predict(x_test)

In [117]:
print(classification_report(y_test,y_pred_dt))

              precision    recall  f1-score   support

           0       0.85      0.87      0.86      1051
           1       0.59      0.54      0.57       358

    accuracy                           0.79      1409
   macro avg       0.72      0.71      0.71      1409
weighted avg       0.78      0.79      0.78      1409



In [118]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


## Initial Insights

- Base model has a accuracy of 76% which is not reliable because of the imbalanced dataset
- TotalCharges needs to be a float/int type (Data Cleaning)
- Need to perform Feature Scaling

## Data Cleaning

In [119]:
telco_data=df.copy()

In [120]:
telco_data.TotalCharges=pd.to_numeric(telco_data.TotalCharges,errors='coerce')

In [121]:
telco_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [122]:
telco_data.loc[telco_data['TotalCharges'].isnull()==True]

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,No,Yes,Yes,Yes,No,Two year,Yes,Bank transfer (automatic),52.55,NaN,No
753,3115-CZMZD,Male,0,No,Yes,0,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.25,NaN,No
936,5709-LVOEQ,Female,0,Yes,Yes,0,Yes,No,DSL,Yes,Yes,Yes,No,Yes,Yes,Two year,No,Mailed check,80.85,NaN,No
1082,4367-NUYAO,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.75,NaN,No
1340,1371-DWPAZ,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,Yes,Yes,Yes,Yes,No,Two year,No,Credit card (automatic),56.05,NaN,No
3331,7644-OMVMY,Male,0,Yes,Yes,0,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,19.85,NaN,No
3826,3213-VVOLG,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.35,NaN,No
4380,2520-SGTTA,Female,0,Yes,Yes,0,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.00,NaN,No
5218,2923-ARZLG,Male,0,Yes,Yes,0,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,One year,Yes,Mailed check,19.70,NaN,No
6670,4075-WKNIU,Female,0,Yes,Yes,0,Yes,Yes,DSL,No,Yes,Yes,Yes,Yes,No,Two year,No,Mailed check,73.35,NaN,No


In [123]:
telco_data.dropna(how='any',inplace=True)

In [124]:
telco_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7032 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7032 non-null   object 
 1   gender            7032 non-null   object 
 2   SeniorCitizen     7032 non-null   int64  
 3   Partner           7032 non-null   object 
 4   Dependents        7032 non-null   object 
 5   tenure            7032 non-null   int64  
 6   PhoneService      7032 non-null   object 
 7   MultipleLines     7032 non-null   object 
 8   InternetService   7032 non-null   object 
 9   OnlineSecurity    7032 non-null   object 
 10  OnlineBackup      7032 non-null   object 
 11  DeviceProtection  7032 non-null   object 
 12  TechSupport       7032 non-null   object 
 13  StreamingTV       7032 non-null   object 
 14  StreamingMovies   7032 non-null   object 
 15  Contract          7032 non-null   object 
 16  PaperlessBilling  7032 non-null   object 
 17  

In [125]:
telco_data.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [126]:
telco_data.tenure.max()

np.int64(72)

In [127]:
bins=[0,12,24,36,48,60,72]
labels=['1-12','13-24','25-36','37-48','49-60','61-72']
telco_data['tenure_bin']=pd.cut(telco_data['tenure'],bins=bins,labels=labels,include_lowest=True)


print(telco_data[['tenure','tenure_bin']].head(10))
print(telco_data['tenure_bin'].value_counts().sort_index())

   tenure tenure_bin
0       1       1-12
1      34      25-36
2       2       1-12
3      45      37-48
4       2       1-12
5       8       1-12
6      22      13-24
7      10       1-12
8      28      25-36
9      62      61-72
tenure_bin
1-12     2175
13-24    1024
25-36     832
37-48     762
49-60     832
61-72    1407
Name: count, dtype: int64


In [128]:
telco_data.head(5)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,tenure_bin
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,1-12
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No,25-36
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1-12
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,37-48
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1-12


In [129]:
y=telco_data['Churn']
x=telco_data.drop(columns=['customerID','Churn','tenure'])


print("Shape of x :",x.shape)
print("shape of y:",y.shape)

Shape of x : (7032, 19)
shape of y: (7032,)


In [130]:
x

,gender,SeniorCitizen,Partner,Dependents,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,tenure_bin
0,Female,0,Yes,No,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,1-12
1,Male,0,No,No,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,25-36
2,Male,0,No,No,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1-12
3,Male,0,No,No,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,37-48
4,Female,0,No,No,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1-12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,Male,0,Yes,Yes,Yes,Yes,DSL,Yes,No,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,84.80,1990.50,13-24
7039,Female,0,Yes,Yes,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.20,7362.90,61-72
7040,Female,0,Yes,Yes,No,No phone service,DSL,Yes,No,No,No,No,No,Month-to-month,Yes,Electronic check,29.60,346.45,1-12
7041,Male,1,Yes,No,Yes,Yes,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Mailed check,74.40,306.60,1-12


In [131]:
x=pd.get_dummies(x,drop_first=True)
y=telco_data['Churn'].map({'No':0,'Yes':1})

In [132]:
x

,SeniorCitizen,MonthlyCharges,TotalCharges,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,MultipleLines_Yes,InternetService_Fiber optic,InternetService_No,OnlineSecurity_No internet service,OnlineSecurity_Yes,OnlineBackup_No internet service,OnlineBackup_Yes,DeviceProtection_No internet service,DeviceProtection_Yes,TechSupport_No internet service,TechSupport_Yes,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,tenure_bin_13-24,tenure_bin_25-36,tenure_bin_37-48,tenure_bin_49-60,tenure_bin_61-72
0,0,29.85,29.85,False,True,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,True,False,False,False,False,False,False
1,0,56.95,1889.50,True,False,False,True,False,False,False,False,False,True,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,True,False,True,False,False,False
2,0,53.85,108.15,True,False,False,True,False,False,False,False,False,True,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,True,False,False,False,False,False
3,0,42.30,1840.75,True,False,False,False,True,False,False,False,False,True,False,False,False,True,False,True,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False
4,0,70.70,151.65,False,False,False,True,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,True,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,0,84.80,1990.50,True,True,True,True,False,True,False,False,False,True,False,False,False,True,False,True,False,True,False,True,True,False,True,False,False,True,True,False,False,False,False
7039,0,103.20,7362.90,False,True,True,True,False,True,True,False,False,False,False,True,False,True,False,False,False,True,False,True,True,False,True,True,False,False,False,False,False,False,True
7040,0,29.60,346.45,False,True,True,False,True,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,True,False,False,False,False,False,False
7041,1,74.40,306.60,True,True,False,True,False,True,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,True,False,False,False,False,False


In [133]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2)

## Feature Scaling

In [134]:
x_train=sc.fit_transform(x_train)
x_test=sc.transform(x_test)

In [135]:
model_dt2.fit(x_train,y_train)

y_pred_dt2=model_dt2.predict(x_test)

In [136]:
print(classification_report(y_test,y_pred_dt2))

              precision    recall  f1-score   support

           0       0.81      0.79      0.80      1017
           1       0.48      0.51      0.49       390

    accuracy                           0.71      1407
   macro avg       0.64      0.65      0.64      1407
weighted avg       0.71      0.71      0.71      1407



## Feature Scaling - MinMaxScaler

In [137]:
x_train_mns=mns.fit_transform(x_train)
x_test_mns=mns.transform(x_test)

In [138]:
model_dt3.fit(x_train_mns,y_train)

y_pred_dt3=model_dt3.predict(x_test_mns)

In [139]:
print(classification_report(y_test,y_pred_dt3))

              precision    recall  f1-score   support

           0       0.81      0.79      0.80      1017
           1       0.48      0.51      0.49       390

    accuracy                           0.71      1407
   macro avg       0.64      0.65      0.65      1407
weighted avg       0.72      0.71      0.71      1407



## SMOTEENN () [UpSampling + ENN)

In [140]:
x_train_resampled,y_train_resampled=smenn.fit_resample(x_train,y_train)

In [141]:
model_dt_smotenn.fit(x_train_resampled,y_train_resampled)

y_pred_dt_smotenn=model_dt_smotenn.predict(x_test)

In [142]:
print(classification_report(y_test,y_pred_dt_smotenn))

              precision    recall  f1-score   support

           0       0.89      0.70      0.78      1017
           1       0.49      0.77      0.60       390

    accuracy                           0.72      1407
   macro avg       0.69      0.73      0.69      1407
weighted avg       0.78      0.72      0.73      1407



In [143]:
model_rf_smotenn.fit(x_train_resampled,y_train_resampled)

y_pred_rf_smotenn=model_rf_smotenn.predict(x_test)

In [144]:
print(classification_report(y_test,y_pred_rf_smotenn))

              precision    recall  f1-score   support

           0       0.91      0.71      0.80      1017
           1       0.52      0.81      0.63       390

    accuracy                           0.74      1407
   macro avg       0.71      0.76      0.71      1407
weighted avg       0.80      0.74      0.75      1407



## XGBoost without SMOTEENN

In [145]:
model_xgb.fit(x_train,y_train)

y_pred_xgb=model_xgb.predict(x_test)

In [146]:
print(classification_report(y_test,y_pred_xgb))

              precision    recall  f1-score   support

           0       0.83      0.88      0.85      1017
           1       0.63      0.52      0.57       390

    accuracy                           0.78      1407
   macro avg       0.73      0.70      0.71      1407
weighted avg       0.77      0.78      0.77      1407



## XGBoost with SMOTEENN

In [147]:
model_xgb_smotenn.fit(x_train_resampled,y_train_resampled)

y_pred_xgb_smotenn=model_xgb_smotenn.predict(x_test)

In [148]:
print(classification_report(y_test,y_pred_xgb_smotenn))

              precision    recall  f1-score   support

           0       0.90      0.73      0.80      1017
           1       0.52      0.79      0.63       390

    accuracy                           0.74      1407
   macro avg       0.71      0.76      0.72      1407
weighted avg       0.80      0.74      0.76      1407



## XGBoost with SMOTE

In [152]:
print("Before Counter ",Counter(y_train))

x_train_smote,y_train_smote=smote.fit_resample(x_train,y_train)

print("After Smote ",Counter(y_train_smote))

model_xgb_smote.fit(x_train_smote,y_train_smote)


y_pred_xgb_smote=model_xgb_smote.predict(x_test)

Before Counter  Counter({0: 4146, 1: 1479})
After Smote  Counter({0: 4146, 1: 4146})


In [153]:
print(classification_report(y_test,y_pred_xgb_smote))

              precision    recall  f1-score   support

           0       0.83      0.86      0.85      1017
           1       0.60      0.56      0.58       390

    accuracy                           0.78      1407
   macro avg       0.72      0.71      0.71      1407
weighted avg       0.77      0.78      0.77      1407



In [163]:
print("Before Adaysn : ",Counter(y_train))

x_train_adasyn,y_train_adasyn=adasyn.fit_resample(x_train,y_train)

print("After Adasyn : ",Counter(y_train_adasyn))

model_xgb_adasyn.fit(x_train_adasyn,y_train_adasyn)
y_pred_xgb_adasyn=model_xgb_adasyn.predict(x_test)


print("Accuracy :",accuracy_score(y_test,y_pred_xgb_adasyn))
print("Classification Report : \n",classification_report(y_test,y_pred_xgb_adasyn))
print("confusion Matrix: \n",confusion_matrix(y_test,y_pred_xgb_adasyn))

Before Adaysn :  Counter({0: 4146, 1: 1479})
After Adasyn :  Counter({0: 4146, 1: 4056})
Accuracy : 0.7739872068230277
Classification Report : 
               precision    recall  f1-score   support

           0       0.83      0.86      0.85      1017
           1       0.60      0.55      0.58       390

    accuracy                           0.77      1407
   macro avg       0.72      0.71      0.71      1407
weighted avg       0.77      0.77      0.77      1407

confusion Matrix: 
 [[873 144]
 [174 216]]


In [171]:
scale_pos_weight=(y_train==0).sum()/(y_train==1).sum()
print(f"Scale Pos Weight : {scale_pos_weight}")

model_xgb_weighted=XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric='logloss'
)

model_xgb_weighted.fit(x_train,y_train)
y_pred_xgb_weighted=model_xgb_weighted.predict(x_test)


print("Accuracy Score is : ",accuracy_score(y_test,y_pred_xgb_weighted))
print("\n Classification Report is \n",classification_report(y_test,y_pred_xgb_weighted))
print("Confusion Matrix :\n",confusion_matrix(y_test,y_pred_xgb_weighted))

Scale Pos Weight : 2.8032454361054766
Accuracy Score is :  0.7583511016346838

 Classification Report is 
               precision    recall  f1-score   support

           0       0.86      0.80      0.83      1017
           1       0.55      0.66      0.60       390

    accuracy                           0.76      1407
   macro avg       0.71      0.73      0.71      1407
weighted avg       0.77      0.76      0.76      1407

Confusion Matrix :
 [[810 207]
 [133 257]]


In [180]:
negative_count=(y_train==0).sum()
positive_count=(y_train==1).sum()
weighted_positive=negative_count/positive_count

print(f"Negative Class Count : {negative_count}")
print(f"Positive Class Count : {positive_count}")
print(f"Weighted For Positive Class : {weighted_positive}")


sample_weight=np.where(y_train==1,weighted_positive,1.0)

model_ada_weighted=AdaBoostClassifier(
    n_estimators=50,
    random_state=42
)

model_ada_weighted.fit(x_train,y_train,sample_weight=sample_weight)

y_pred_ada_weighted=model_ada_weighted.predict(x_test)

print("Adaboost with Sample Weighted Accuracy is : ",accuracy_score(y_test,y_pred_ada_weighted))
print("Classification Report is : \n",classification_report(y_test,y_pred_ada_weighted))
print("\n Confusion Matrix \n",confusion_matrix(y_test,y_pred_ada_weighted))

Negative Class Count : 4146
Positive Class Count : 1479
Weighted For Positive Class : 2.8032454361054766
Adaboost with Sample Weighted Accuracy is :  0.7533759772565742
Classification Report is : 
               precision    recall  f1-score   support

           0       0.90      0.74      0.81      1017
           1       0.54      0.78      0.64       390

    accuracy                           0.75      1407
   macro avg       0.72      0.76      0.72      1407
weighted avg       0.80      0.75      0.76      1407


 Confusion Matrix 
 [[756 261]
 [ 86 304]]


## **Hyper Parameter Optimization**

In [184]:
param_grid={
    'max_depth': [3, 4, 5, 6, 7],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'n_estimators': [100, 200, 300, 400],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'min_child_weight': [1, 3, 5],
    'gamma': [0, 0.1, 0.2],
    'scale_pos_weight': [1, scale_pos_weight] 

}
xgb=XGBClassifier(random_state=42,eval_metric='logloss')

search=RandomizedSearchCV(
    xgb,
    param_grid,
    n_iter=50,
    cv=5,
    scoring='f1',
    random_state=42,
    n_jobs=-1

)


search.fit(x_train,y_train)
print("Best Parameters : ",search.best_params_)
print("Best Cv F1 Score ",search.best_score_)

best_model=search.best_estimator_
y_pred_best=best_model.predict(x_test)
print("\n Classification Report is \n",classification_report(y_test,y_pred_best))

Best Parameters :  {'subsample': 0.8, 'scale_pos_weight': np.float64(2.8032454361054766), 'n_estimators': 200, 'min_child_weight': 5, 'max_depth': 4, 'learning_rate': 0.05, 'gamma': 0.2, 'colsample_bytree': 0.8}
Best Cv F1 Score  0.6293339972808902

 Classification Report is 
               precision    recall  f1-score   support

           0       0.90      0.74      0.81      1017
           1       0.54      0.78      0.64       390

    accuracy                           0.75      1407
   macro avg       0.72      0.76      0.73      1407
weighted avg       0.80      0.75      0.76      1407



In [187]:
joblib.dump(best_model,'../models/best_xgboost_churn_model.pkl')
print("Best model Successfully saved as 'best_xgboost_churn_model.pkl' ")

Best model Successfully saved as 'best_xgboost_churn_model.pkl' 


## Trying out few more techniques

In [193]:
model_rf.fit(x_train,y_train)
y_pred_rf=model_rf.predict(x_test)

print(classification_report(y_test,y_pred_rf))

              precision    recall  f1-score   support

           0       0.89      0.72      0.80      1017
           1       0.51      0.77      0.62       390

    accuracy                           0.73      1407
   macro avg       0.70      0.75      0.71      1407
weighted avg       0.79      0.73      0.75      1407



In [194]:
model_cat.fit(x_train,y_train)
y_pred_cat=model_cat.predict(x_test)
print(classification_report(y_test,y_pred_cat))

              precision    recall  f1-score   support

           0       0.88      0.76      0.82      1017
           1       0.54      0.73      0.62       390

    accuracy                           0.75      1407
   macro avg       0.71      0.75      0.72      1407
weighted avg       0.79      0.75      0.76      1407



In [200]:
model_lgb=LGBMClassifier(
    scale_pos_weight=scale_pos_weight,
    random_state=42
)
model_lgb.fit(x_train,y_train)
y_pred_lgb=model_lgb.predict(x_test)
print(classification_report(y_test,y_pred_lgb))

              precision    recall  f1-score   support

           0       0.88      0.77      0.82      1017
           1       0.54      0.72      0.62       390

    accuracy                           0.75      1407
   macro avg       0.71      0.74      0.72      1407
weighted avg       0.78      0.75      0.76      1407



## **Optuna fine tuning**

In [203]:
def objective(trial):
    params={
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0, 0.5),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 1),  # L1 regularization
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 2), # L2 regularization
        'scale_pos_weight': trial.suggest_categorical('scale_pos_weight', [1, scale_pos_weight]),
        'random_state': 42,
        'eval_metric': 'logloss'

    }
    model=XGBClassifier(**params)

    f1_scorer=make_scorer(f1_score,average='binary',pos_label=1)
    score=cross_val_score(model,x_train,y_train,cv=5,scoring=f1_scorer,n_jobs=-1).mean()
    return score

study=optuna.create_study(direction='maximize')
study.optimize(objective,n_trials=100)
print("Best Parameters ",study.best_params)
print("Best CV F1 Score : ",study.best_value)

best_model_optuna=XGBClassifier(**study.best_params)
best_model_optuna.fit(x_train,y_train)
y_pred_optuna=best_model_optuna.predict(x_test)
print(classification_report(y_test,y_pred_optuna))



[I 2026-04-02 00:32:35,122] A new study created in memory with name: no-name-ac333e5c-280f-4725-8049-d84a69da6865
[I 2026-04-02 00:32:37,831] Trial 0 finished with value: 0.629252729482298 and parameters: {'max_depth': 7, 'learning_rate': 0.011779501811490874, 'n_estimators': 256, 'subsample': 0.8424717203688067, 'colsample_bytree': 0.8846222229067306, 'min_child_weight': 1, 'gamma': 0.3035622121240465, 'reg_alpha': 0.708132289641513, 'reg_lambda': 1.8477871460748658, 'scale_pos_weight': np.float64(2.8032454361054766)}. Best is trial 0 with value: 0.629252729482298.
[I 2026-04-02 00:32:39,991] Trial 1 finished with value: 0.5816361113518731 and parameters: {'max_depth': 5, 'learning_rate': 0.028074273844526123, 'n_estimators': 330, 'subsample': 0.8690493057221444, 'colsample_bytree': 0.7670059225747811, 'min_child_weight': 6, 'gamma': 0.4290770047483449, 'reg_alpha': 0.7372080287957464, 'reg_lambda': 1.977525444348415, 'scale_pos_weight': 1}. Best is trial 0 with value: 0.6292527294822

Best Parameters  {'max_depth': 7, 'learning_rate': 0.016185643993294006, 'n_estimators': 192, 'subsample': 0.873483782608229, 'colsample_bytree': 0.8126365338356265, 'min_child_weight': 2, 'gamma': 0.48632100527915945, 'reg_alpha': 0.26988138342930335, 'reg_lambda': 1.698154300691567, 'scale_pos_weight': np.float64(2.8032454361054766)}
Best CV F1 Score :  0.6304188049947953
              precision    recall  f1-score   support

           0       0.89      0.75      0.81      1017
           1       0.54      0.76      0.63       390

    accuracy                           0.75      1407
   macro avg       0.72      0.76      0.72      1407
weighted avg       0.79      0.75      0.76      1407



In [204]:
joblib.dump(best_model_optuna,'../models/best_optuna_churn_model.pkl')
print("Best Model Succesfully saved as 'best_optuna_churn_model.pkl' ")

Best Model Succesfully saved as 'best_optuna_churn_model.pkl' 


In [205]:
joblib.dump(model_ada_weighted,'../models/ada_boost_churn_model.pkl')
print("best model  succesfully saved as 'ada_boost_churn_model.pkl'")

best model  succesfully saved as 'ada_boost_churn_model.pkl'
